In [5]:
%run -i mine_bbh_golden_qwen.py

Setting device natively...
Loading model natively...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

ruin_names/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.2kB            

ruin_names/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]



































































































ruin_names:  39%|███▉      | 97/248 [01:50<02:51,  1.13s/it]


ValueError: Cannot extract native letter from: 'dearth, wind, & fire'

In [6]:
import json
import random
from collections import defaultdict, Counter

print("\n\n" + "="*60)
print(f"SALVAGING MINED DATA. Found {len(golden_records)} total raw golden records before crash.")
print("Performing strict mathematical distribution balancing...")

MC_LETTERS_ALL = ["A", "B", "C", "D", "E", "F"]
rng = random.Random(42)

binary_records = defaultdict(list)
multi_records = defaultdict(list)

for r in golden_records:
    ans = r["answer"]
    if len(r["valid_letters"]) == 2:
        binary_records[ans].append(r)
    else:
        multi_records[ans].append(r)

final_dataset = []

if binary_records:
    binary_keys = ["A", "B"]
    min_binary = min([len(binary_records.get(k, [])) for k in binary_keys])
    print(f"Binary Tasks: Smallest class has {min_binary} records. Truncating A and B to {min_binary}.")
    for k in binary_keys:
        if k in binary_records:
            final_dataset.extend(rng.sample(binary_records[k], min_binary))

if multi_records:
    active_keys = [k for k in MC_LETTERS_ALL if k in multi_records]
    min_multi = min([len(multi_records[k]) for k in active_keys])
    print(f"Multi-Choice Tasks: Smallest class has {min_multi} records. Truncating {', '.join(active_keys)} to {min_multi}.")
    for k in active_keys:
        final_dataset.extend(rng.sample(multi_records[k], min_multi))

rng.shuffle(final_dataset)

with open("bbh_golden_v2.json", "w") as f:
    json.dump(final_dataset, f, indent=2)

print("\nFinal Balanced Dataset saved to 'bbh_golden_v2.json'.")
print(f"Total Size: {len(final_dataset)} records.")

print("\nFinal Binary Distribution:")
print(Counter([r['answer'] for r in final_dataset if len(r['valid_letters'])==2]))

print("\nFinal Multi-Choice Distribution:")
print(Counter([r['answer'] for r in final_dataset if len(r['valid_letters'])>2]))



SALVAGING MINED DATA. Found 180 total raw golden records before crash.
Performing strict mathematical distribution balancing...
Binary Tasks: Smallest class has 14 records. Truncating A and B to 14.
Multi-Choice Tasks: Smallest class has 2 records. Truncating A, B, C, D, E, F to 2.

Final Balanced Dataset saved to 'bbh_golden_v2.json'.
Total Size: 40 records.

Final Binary Distribution:
Counter({'B': 14, 'A': 14})

Final Multi-Choice Distribution:
Counter({'C': 2, 'B': 2, 'E': 2, 'A': 2, 'D': 2, 'F': 2})


In [8]:
%run -i cache_teacher.py

Device: cuda
Loading tokenizer and teacher model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading bbh_golden_v2.json...
Dataset has 40 records. Caching teacher states for 32 training records.
Pre-computing teacher hidden states...


Teacher targets: 100%|██████████| 32/32 [00:21<00:00,  1.47it/s]

✅ Saved 32 pairs to teacher_hidden_states.pt
🔥 CRITICAL: Restart your Colab Kernel / Runtime NOW to completely free VRAM before training!


In [9]:
import json

with open("bbh_golden_raw_180.json", "w") as f:
    json.dump(golden_records, f, indent=2)

print(f"Saved all {len(golden_records)} raw records to bbh_golden_raw_180.json!")

Saved all 40 raw records to bbh_golden_raw_180.json!


In [10]:
%run -i train_adapter_v2.py

Purging leftover models from VRAM...
Device: cuda
Tokenizer already loaded.
Loading student model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Dynamic Split: 32 train / 8 test
Running 0-shot (Student) and 2-shot (Teacher) baselines on test set...


Baseline: 100%|██████████| 8/8 [00:06<00:00,  1.28it/s]


Loading pre-computed teacher states from disk...
Loaded 32 pairs with 37 layers into VRAM.
Adapter: freshly injected into student model.
Trainable scalars: 589,824
No checkpoint found — starting fresh from Epoch 1.

Training Epochs 1 → 10
Epoch 001 | Loss: 3.4161  CE: 3.4072  Cos: 0.0889  L2: 236.7
Epoch 002 | Loss: 2.0153  CE: 2.0060  Cos: 0.0935  L2: 237.3
Epoch 003 | Loss: 1.1706  CE: 1.1602  Cos: 0.1049  L2: 238.5
Epoch 004 | Loss: 0.9457  CE: 0.9329  Cos: 0.1274  L2: 240.2
Epoch 005 | Loss: 0.8176  CE: 0.8032  Cos: 0.1440  L2: 241.8
Epoch 006 | Loss: 0.6595  CE: 0.6434  Cos: 0.1606  L2: 243.6
Epoch 007 | Loss: 0.5766  CE: 0.5593  Cos: 0.1727  L2: 245.5
Epoch 008 | Loss: 0.5100  CE: 0.4921  Cos: 0.1790  L2: 247.4
Epoch 009 | Loss: 0.4635  CE: 0.4449  Cos: 0.1861  L2: 249.4
Epoch 010 | Loss: 0.4246  CE: 0.4048  Cos: 0.1980  L2: 251.4

Checkpoint saved → adapter_epoch_010.pt

EVALUATION AT EPOCH 10
ID     Task                           Ans  | Tch(2S) | Base(0S) |  Now(0S)
-----------

In [11]:
%run -i train_adapter_v2.py

Purging leftover models from VRAM...
Device: cuda
Tokenizer already loaded.
Student model already loaded.
Dynamic Split: 32 train / 8 test
Baseline already cached.
Loading pre-computed teacher states from disk...
Loaded 32 pairs with 37 layers into VRAM.
Adapter: already in model — reusing existing layers.
Trainable scalars: 589,824
Resumed from adapter_epoch_010.pt — starting at Epoch 11.

Training Epochs 11 → 20
Epoch 011 | Loss: 0.3754  CE: 0.3544  Cos: 0.2097  L2: 253.4
Epoch 012 | Loss: 0.3477  CE: 0.3255  Cos: 0.2218  L2: 255.5
Epoch 013 | Loss: 0.3287  CE: 0.3049  Cos: 0.2381  L2: 257.5
Epoch 014 | Loss: 0.3685  CE: 0.3426  Cos: 0.2588  L2: 259.6
Epoch 015 | Loss: 0.3390  CE: 0.3123  Cos: 0.2666  L2: 261.6
Epoch 016 | Loss: 0.2798  CE: 0.2529  Cos: 0.2689  L2: 263.6
Epoch 017 | Loss: 0.2243  CE: 0.1967  Cos: 0.2764  L2: 265.5
Epoch 018 | Loss: 0.2237  CE: 0.1950  Cos: 0.2876  L2: 267.4
Epoch 019 | Loss: 0.4565  CE: 0.4270  Cos: 0.2954  L2: 269.3
Epoch 020 | Loss: 0.4568  CE: 0.4

In [13]:
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

print("1. Loading datasets and filtering out used records...")
with open("bbh_golden_raw_180.json", "r") as f:
    raw_records = json.load(f)
with open("bbh_golden_v2.json", "r") as f:
    used_records = json.load(f)

used_ids = set(r["id"] for r in used_records)
available_pool = [r for r in raw_records if r["id"] not in used_ids]

print(f"Total raw records: {len(raw_records)}")
print(f"Used in previous training/testing: {len(used_ids)}")
print(f"Available for blind test: {len(available_pool)}")

random.seed(99) 
blind_test = random.sample(available_pool, min(30, len(available_pool)))
print(f"Sampled {len(blind_test)} strictly unseen tasks for evaluation.\n")

def predict_mc(prompt: str, valid_letters: list):
    formatted = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True) + "Answer:"
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = student_model(**inputs).logits[:, -1, :].float()
        probs = F.softmax(logits, dim=-1)
    
    MC_TOKEN_IDS = {l: tokenizer.encode(l, add_special_tokens=False)[-1] for l in valid_letters}
    raw = [probs[0, MC_TOKEN_IDS[l]].item() for l in valid_letters]
    total = sum(raw)
    renorm = [r / total for r in raw] if total > 0 else [0]*len(valid_letters)
    best = renorm.index(max(renorm))
    return valid_letters[best]

# ---------------------------------------------------------
# STRIP ADAPTER FOR PURE BASELINE
# ---------------------------------------------------------
if hasattr(student_model.model.layers[0].self_attn, 'original_attn'):
    print("Stripping the adapter to evaluate pure Base model...")
    for layer in student_model.model.layers:
        layer.self_attn = layer.self_attn.original_attn

print("2. Evaluating Base (0-Shot) and Teacher (2-Shot) on unseen data...")
results = {}
for t in blind_test:
    ans = t["answer"]
    vl = t["valid_letters"]
    s_pred = predict_mc(t["student_prompt"], vl)
    t_pred = predict_mc(t["teacher_prompt"], vl)
    results[t["id"]] = {
        "task": t["task"], "ans": ans, 
        "base_0s": s_pred, "tch_2s": t_pred
    }

# ---------------------------------------------------------
# CALIBRATED ATTENTION DEFINITION
# ---------------------------------------------------------
def _rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def _apply_rope(q, k, cos, sin):
    if cos.dim() == 3:
        cos, sin = cos.unsqueeze(1), sin.unsqueeze(1)
    return (q * cos) + (_rotate_half(q) * sin), (k * cos) + (_rotate_half(k) * sin)

class CalibratedAttention(nn.Module):
    def __init__(self, original_attn, rank: int = 4):
        super().__init__()
        self.original_attn    = original_attn
        cfg                   = original_attn.config
        self.num_heads        = cfg.num_attention_heads
        self.num_kv_heads     = getattr(cfg, "num_key_value_heads", self.num_heads)
        self.hidden_size      = cfg.hidden_size
        self.head_dim         = self.hidden_size // self.num_heads
        self.num_kv_groups    = self.num_heads // self.num_kv_heads
        self._tuple_len       = None
        self.U_q = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        self.U_k = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        nn.init.normal_(self.U_q, std=0.02)
        nn.init.normal_(self.U_k, std=0.02)

    def forward(self, hidden_states, attention_mask=None, position_ids=None, past_key_value=None, output_attentions=False, use_cache=False, cache_position=None, position_embeddings=None, **kwargs):
        if self._tuple_len is None:
            with torch.no_grad():
                dummy = self.original_attn(hidden_states=hidden_states, attention_mask=attention_mask, position_ids=position_ids, past_key_value=past_key_value, output_attentions=output_attentions, use_cache=use_cache, cache_position=cache_position, position_embeddings=position_embeddings, **kwargs)
                self._tuple_len = len(dummy) if isinstance(dummy, tuple) else 1
        B, T, _ = hidden_states.shape
        with torch.no_grad():
            Q = self.original_attn.q_proj(hidden_states)
            K = self.original_attn.k_proj(hidden_states)
            V = self.original_attn.v_proj(hidden_states)
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        with torch.no_grad():
            if position_embeddings is not None:
                cos, sin = position_embeddings
            else:
                cos, sin = self.original_attn.rotary_emb(V, position_ids)
            Q, K = _apply_rope(Q, K, cos.float(), sin.float())
            if self.num_kv_groups > 1:
                K = K.repeat_interleave(self.num_kv_groups, dim=1)
                V = V.repeat_interleave(self.num_kv_groups, dim=1)
        Q, K, V = Q.detach().float(), K.detach().float(), V.detach().float()
        A     = torch.einsum('bhtd,hdr->bhtr', Q, self.U_q)
        Bm    = torch.einsum('bhtd,hdr->bhtr', K, self.U_k)
        delta = torch.matmul(A, Bm.transpose(-1, -2))
        with torch.no_grad():
            S      = torch.matmul(Q, K.transpose(-1, -2)) / (self.head_dim ** 0.5)
            causal = torch.tril(torch.ones(T, T, device=device)).bool()
            S      = S.masked_fill(~causal, torch.finfo(S.dtype).min)
        S_prime = S + delta
        if attention_mask is not None:
            S_prime = S_prime + attention_mask.float()
        attn_weights = F.softmax(S_prime, dim=-1)
        attn_output  = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.head_dim).to(dtype=hidden_states.dtype)
        return (self.original_attn.o_proj(attn_output), None, None, None)[:self._tuple_len]

# ---------------------------------------------------------
# INJECT ADAPTER AND EVALUATE
# ---------------------------------------------------------
print("\n3. Injecting Adapter and loading weights from adapter_epoch_020.pt...")
trainable_params = []
for layer in student_model.model.layers:
    cal = CalibratedAttention(layer.self_attn, rank=4).to(device)
    layer.self_attn = cal
    trainable_params.extend([cal.U_q, cal.U_k])

ckpt = torch.load("adapter_epoch_020.pt", map_location=device)
for p, saved in zip(trainable_params, ckpt["adapter_weights"]):
    p.data.copy_(saved.to(device))

student_model.eval()

print("4. Evaluating Trained Student (0-Shot) on unseen data...")
for t in blind_test:
    vl = t["valid_letters"]
    trained_pred = predict_mc(t["student_prompt"], vl)
    results[t["id"]]["trained_0s"] = trained_pred

# ---------------------------------------------------------
# PRINT RESULTS
# ---------------------------------------------------------
base_correct = 0
tch_correct = 0
trained_correct = 0
intersection_all = 0
distilled_success = 0 

print(f"\n{'='*100}")
print(f"{'ID':<6} {'Task':<28} {'Ans':>3} | {'Tch(2S)':>7} | {'Base(0S)':>8} | {'Trn(0S)':>8}")
print("-" * 100)

for t in blind_test:
    r = results[t["id"]]
    ans = r["ans"]
    
    t_corr = r["tch_2s"] == ans
    b_corr = r["base_0s"] == ans
    trn_corr = r["trained_0s"] == ans
    
    tch_correct += t_corr
    base_correct += b_corr
    trained_correct += trn_corr
    
    if t_corr and b_corr and trn_corr:
        intersection_all += 1
    if t_corr and trn_corr and not b_corr:
        distilled_success += 1
        
    t_mark = "✅" if t_corr else "❌"
    b_mark = "✅" if b_corr else "❌"
    trn_mark = "✅" if trn_corr else "❌"
    
    change = ""
    if not b_corr and trn_corr:
        change = "← DISTILLED ✨"
    elif b_corr and not trn_corr:
        change = "← REGRESSED ⚠️"
        
    print(f"{t['id']:<6} {r['task'][:26]:<28} {ans:>3} | {r['tch_2s']} {t_mark}   | {r['base_0s']} {b_mark}    | {r['trained_0s']} {trn_mark}    {change}")

print("-" * 100)
total = len(blind_test)
print(f"\nTeacher (2-shot)      : {tch_correct}/{total} ({tch_correct/total:.0%})")
print(f"Base Student (0-shot) : {base_correct}/{total} ({base_correct/total:.0%})")
print(f"Trained (0-shot)      : {trained_correct}/{total} ({trained_correct/total:.0%})")

print(f"\nIntersection (Correct in ALL THREE) : {intersection_all}")
print(f"True Distillation (Wrong in Base, Correct in Teacher & Trained): {distilled_success}")

1. Loading datasets and filtering out used records...
Total raw records: 40
Used in previous training/testing: 40
Available for blind test: 0
Sampled 0 strictly unseen tasks for evaluation.

Stripping the adapter to evaluate pure Base model...
2. Evaluating Base (0-Shot) and Teacher (2-Shot) on unseen data...

3. Injecting Adapter and loading weights from adapter_epoch_020.pt...
4. Evaluating Trained Student (0-Shot) on unseen data...

ID     Task                         Ans | Tch(2S) | Base(0S) |  Trn(0S)
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------


ZeroDivisionError: division by zero

In [14]:
import json

with open("bbh_golden_raw_180.json", "r") as f:
    raw_records = json.load(f)
    
with open("bbh_golden_v2.json", "r") as f:
    used_records = json.load(f)

print(f"Total raw records inside the 180 file: {len(raw_records)}")
print(f"Total used records inside the v2 file: {len(used_records)}")

Total raw records inside the 180 file: 40
Total used records inside the v2 file: 40


In [15]:
import json
import re
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tqdm import tqdm

print("==================================================================================")
print("TRUE GENERALIZATION EVALUATION")
print("Fetching entirely unseen REAL records from the tail end of HuggingFace BBH datasets...")
print("==================================================================================")

MC_LETTERS_ALL = ["A", "B", "C", "D", "E", "F"]

if 'tokenizer' not in globals():
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if 'student_model' in globals():
    eval_model = globals()['student_model']
elif 'model' in globals():
    eval_model = globals()['model']
else:
    raise RuntimeError("Could not find 'student_model' or 'model' in memory.")

MC_TOKEN_IDS = {}
for letter in MC_LETTERS_ALL:
    MC_TOKEN_IDS[letter] = tokenizer.encode(letter, add_special_tokens=False)[-1]

rng = random.Random(42)

# Exclude ruin_names due to the earlier parsing issue
TASK_CONFIGS = {
    "boolean_expressions": {"task_desc": "Evaluate the Boolean expression.", "mode": "dynamic", "num_options": 2},
    "navigate": {"task_desc": "Determine whether the navigation instructions lead back to the starting point.", "mode": "dynamic", "num_options": 2},
    "sports_understanding": {"task_desc": "Determine whether the sports statement is plausible or implausible.", "mode": "dynamic", "num_options": 2},
    "tracking_shuffled_objects_three_objects": {"task_desc": "Track the ownership of objects after swaps.", "mode": "dynamic", "num_options": 6},
    "date_understanding": {"task_desc": "Infer the date from the given context.", "mode": "native", "num_options": 6},
    "logical_deduction_five_objects": {"task_desc": "Deduce the logical ordering.", "mode": "native", "num_options": 5},
    "snarks": {"task_desc": "Identify the sarcastic sentence.", "mode": "native", "num_options": 2},
}

def format_prompt(text: str) -> str:
    messages = [{"role": "user", "content": text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Answer:"

def predict_mc(prompt_text: str, valid_letters: list) -> str:
    formatted = format_prompt(prompt_text)
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = eval_model(**inputs).logits[:, -1, :].float()
        probs  = F.softmax(logits, dim=-1)
    
    raw_probs = [probs[0, MC_TOKEN_IDS[l]].item() for l in valid_letters]
    total = sum(raw_probs)
    if total == 0: return valid_letters[0]
    renorm_probs = [p / total for p in raw_probs]
    best_idx = renorm_probs.index(max(renorm_probs))
    return valid_letters[best_idx]

def extract_native_letter(target: str) -> str:
    m = re.search(r'\(([A-F])\)', target)
    if m:
        return m.group(1)
    return None

def build_dynamic_options(correct_target: str, wrong_pool: list, num_options: int, slot: int) -> tuple:
    num_wrong = num_options - 1
    sampled_wrong = rng.sample(wrong_pool, min(num_wrong, len(wrong_pool)))
    options = sampled_wrong[:]
    options.insert(slot, correct_target)
    answer_letter = MC_LETTERS_ALL[slot]
    options_lines = "\n".join(f"{MC_LETTERS_ALL[i]}) {opt}" for i, opt in enumerate(options))
    return f"Options:\n{options_lines}", answer_letter

# =============================================================================
# 1. GENERATE UNSEEN PROMPTS FROM HUGGINGFACE
# =============================================================================
blind_test = []
global_id = 8000

for subset, cfg in TASK_CONFIGS.items():
    mode = cfg["mode"]
    task_desc = cfg["task_desc"]
    num_options = cfg["num_options"]
    valid_letters = MC_LETTERS_ALL[:num_options]

    ds = load_dataset("lukaemon/bbh", subset, split="test")

    if mode == "dynamic":
        unique_pool = list(set([ex["target"] for ex in ds]))

    demos = [ds[0], ds[1]]
    
    # GUARANTEE THEY ARE UNSEEN: Pull exactly 5 records from the very end of the dataset.
    tail_records = ds.select(range(len(ds)-5, len(ds)))

    slot_counter = 0
    demo_str = ""
    for i, demo in enumerate(demos):
        if mode == "dynamic":
            wrong_pool = [t for t in unique_pool if t != demo["target"]]
            opts_txt, ans_letter = build_dynamic_options(demo["target"], wrong_pool, num_options, slot_counter % num_options)
            slot_counter += 1
            demo_str += f"Example {i+1}\n{demo['input']}\n{opts_txt}\nAnswer: {ans_letter}\n\n"
        else:
            ans_letter = extract_native_letter(demo["target"])
            if ans_letter:
                demo_str += f"Example {i+1}\n{demo['input']}\nAnswer: {ans_letter}\n\n"

    for ex in tail_records:
        if mode == "dynamic":
            wrong_pool = [t for t in unique_pool if t != ex["target"]]
            opts_txt, ans_letter = build_dynamic_options(ex["target"], wrong_pool, num_options, slot_counter % num_options)
            slot_counter += 1
            question_block = f"Question\n{ex['input']}\n{opts_txt}"
        else:
            ans_letter = extract_native_letter(ex["target"])
            question_block = f"Question\n{ex['input']}"
            
        if not ans_letter:
            continue

        teacher_prompt = f"Task: {task_desc}\n\n{demo_str}{question_block}"
        student_prompt = f"Task: {task_desc}\n\n{question_block}"

        blind_test.append({
            "id": global_id,
            "task": subset,
            "teacher_prompt": teacher_prompt,
            "student_prompt": student_prompt,
            "answer": ans_letter,
            "valid_letters": valid_letters
        })
        global_id += 1

print(f"\nSuccessfully pulled and formatted {len(blind_test)} purely unseen records.\n")

# =========================================================
# 2. EVALUATE BASE 0-SHOT AND TEACHER 2-SHOT
# =========================================================
if hasattr(eval_model.model.layers[0].self_attn, 'original_attn'):
    print("Stripping the adapter to evaluate pure Base model...")
    for layer in eval_model.model.layers:
        layer.self_attn = layer.self_attn.original_attn

print("Evaluating Base (0-Shot) and Teacher (2-Shot)...")
results = {}
for t in tqdm(blind_test, desc="Baseline & Teacher"):
    ans = t["answer"]
    vl = t["valid_letters"]
    s_pred = predict_mc(t["student_prompt"], vl)
    t_pred = predict_mc(t["teacher_prompt"], vl)
    results[t["id"]] = {
        "task": t["task"], "ans": ans, 
        "base_0s": s_pred, "tch_2s": t_pred
    }

# =========================================================
# 3. INJECT ADAPTER AND EVALUATE TRAINED 0-SHOT
# =========================================================
def _rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def _apply_rope(q, k, cos, sin):
    if cos.dim() == 3:
        cos, sin = cos.unsqueeze(1), sin.unsqueeze(1)
    return (q * cos) + (_rotate_half(q) * sin), (k * cos) + (_rotate_half(k) * sin)

class CalibratedAttention(nn.Module):
    def __init__(self, original_attn, rank: int = 4):
        super().__init__()
        self.original_attn    = original_attn
        cfg                   = original_attn.config
        self.num_heads        = cfg.num_attention_heads
        self.num_kv_heads     = getattr(cfg, "num_key_value_heads", self.num_heads)
        self.hidden_size      = cfg.hidden_size
        self.head_dim         = self.hidden_size // self.num_heads
        self.num_kv_groups    = self.num_heads // self.num_kv_heads
        self._tuple_len       = None
        self.U_q = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        self.U_k = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        nn.init.normal_(self.U_q, std=0.02)
        nn.init.normal_(self.U_k, std=0.02)

    def forward(self, hidden_states, attention_mask=None, position_ids=None, past_key_value=None, output_attentions=False, use_cache=False, cache_position=None, position_embeddings=None, **kwargs):
        if self._tuple_len is None:
            with torch.no_grad():
                dummy = self.original_attn(hidden_states=hidden_states, attention_mask=attention_mask, position_ids=position_ids, past_key_value=past_key_value, output_attentions=output_attentions, use_cache=use_cache, cache_position=cache_position, position_embeddings=position_embeddings, **kwargs)
                self._tuple_len = len(dummy) if isinstance(dummy, tuple) else 1
        B, T, _ = hidden_states.shape
        with torch.no_grad():
            Q = self.original_attn.q_proj(hidden_states)
            K = self.original_attn.k_proj(hidden_states)
            V = self.original_attn.v_proj(hidden_states)
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        with torch.no_grad():
            if position_embeddings is not None:
                cos, sin = position_embeddings
            else:
                cos, sin = self.original_attn.rotary_emb(V, position_ids)
            Q, K = _apply_rope(Q, K, cos.float(), sin.float())
            if self.num_kv_groups > 1:
                K = K.repeat_interleave(self.num_kv_groups, dim=1)
                V = V.repeat_interleave(self.num_kv_groups, dim=1)
        Q, K, V = Q.detach().float(), K.detach().float(), V.detach().float()
        A     = torch.einsum('bhtd,hdr->bhtr', Q, self.U_q)
        Bm    = torch.einsum('bhtd,hdr->bhtr', K, self.U_k)
        delta = torch.matmul(A, Bm.transpose(-1, -2))
        with torch.no_grad():
            S      = torch.matmul(Q, K.transpose(-1, -2)) / (self.head_dim ** 0.5)
            causal = torch.tril(torch.ones(T, T, device=device)).bool()
            S      = S.masked_fill(~causal, torch.finfo(S.dtype).min)
        S_prime = S + delta
        if attention_mask is not None:
            S_prime = S_prime + attention_mask.float()
        attn_weights = F.softmax(S_prime, dim=-1)
        attn_output  = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.head_dim).to(dtype=hidden_states.dtype)
        return (self.original_attn.o_proj(attn_output), None, None, None)[:self._tuple_len]

print("\nInjecting Adapter and loading weights from adapter_epoch_020.pt...")
trainable_params = []
for layer in eval_model.model.layers:
    cal = CalibratedAttention(layer.self_attn, rank=4).to(device)
    layer.self_attn = cal
    trainable_params.extend([cal.U_q, cal.U_k])

ckpt = torch.load("adapter_epoch_020.pt", map_location=device)
for p, saved in zip(trainable_params, ckpt["adapter_weights"]):
    p.data.copy_(saved.to(device))

eval_model.eval()

print("Evaluating Trained Student (0-Shot)...")
for t in tqdm(blind_test, desc="Trained Model"):
    vl = t["valid_letters"]
    trained_pred = predict_mc(t["student_prompt"], vl)
    results[t["id"]]["trained_0s"] = trained_pred

# =========================================================
# 4. PRINT RESULTS
# =========================================================
base_correct = 0
tch_correct = 0
trained_correct = 0
intersection_all = 0
distilled_success = 0 

print(f"\n{'='*100}")
print(f"{'ID':<6} {'Task':<35} {'Ans':>3} | {'Tch(2S)':>7} | {'Base(0S)':>8} | {'Trn(0S)':>8}")
print("-" * 100)

for t in blind_test:
    r = results[t["id"]]
    ans = r["ans"]
    
    t_corr = r["tch_2s"] == ans
    b_corr = r["base_0s"] == ans
    trn_corr = r["trained_0s"] == ans
    
    tch_correct += t_corr
    base_correct += b_corr
    trained_correct += trn_corr
    
    if t_corr and b_corr and trn_corr:
        intersection_all += 1
    if t_corr and trn_corr and not b_corr:
        distilled_success += 1
        
    t_mark = "✅" if t_corr else "❌"
    b_mark = "✅" if b_corr else "❌"
    trn_mark = "✅" if trn_corr else "❌"
    
    change = ""
    if not b_corr and trn_corr:
        change = "← DISTILLED ✨"
    elif b_corr and not trn_corr:
        change = "← REGRESSED ⚠️"
        
    print(f"{t['id']:<6} {r['task'][:33]:<35} {ans:>3} | {r['tch_2s']} {t_mark}   | {r['base_0s']} {b_mark}    | {r['trained_0s']} {trn_mark}    {change}")

print("-" * 100)
total = len(blind_test)
print(f"\nTeacher (2-shot)      : {tch_correct}/{total} ({tch_correct/total:.0%})")
print(f"Base Student (0-shot) : {base_correct}/{total} ({base_correct/total:.0%})")
print(f"Trained (0-shot)      : {trained_correct}/{total} ({trained_correct/total:.0%})")

print(f"\nIntersection (Correct in ALL THREE) : {intersection_all}")
print(f"True Distillation (Wrong in Base, Correct in Teacher & Trained): {distilled_success}")


TRUE GENERALIZATION EVALUATION
Fetching entirely unseen REAL records from the tail end of HuggingFace BBH datasets...

Successfully pulled and formatted 35 purely unseen records.

Stripping the adapter to evaluate pure Base model...
Evaluating Base (0-Shot) and Teacher (2-Shot)...


Baseline & Teacher: 100%|██████████| 35/35 [00:46<00:00,  1.33s/it]



Injecting Adapter and loading weights from adapter_epoch_020.pt...
Evaluating Trained Student (0-Shot)...


Trained Model: 100%|██████████| 35/35 [00:16<00:00,  2.15it/s]


ID     Task                                Ans | Tch(2S) | Base(0S) |  Trn(0S)
----------------------------------------------------------------------------------------------------
8000   boolean_expressions                   A | A ✅   | A ✅    | B ❌    ← REGRESSED ⚠️
8001   boolean_expressions                   B | B ✅   | B ✅    | A ❌    ← REGRESSED ⚠️
8002   boolean_expressions                   A | B ❌   | A ✅    | A ✅    
8003   boolean_expressions                   B | A ❌   | A ❌    | B ✅    ← DISTILLED ✨
8004   boolean_expressions                   A | A ✅   | A ✅    | A ✅    
8005   navigate                              A | A ✅   | A ✅    | A ✅    
8006   navigate                              B | A ❌   | A ❌    | B ✅    ← DISTILLED ✨
8007   navigate                              A | A ✅   | A ✅    | B ❌    ← REGRESSED ⚠️
8008   navigate                              B | B ✅   | B ✅    | B ✅    
8009   navigate                              A | A ✅   | A ✅    | B ❌    ← REGRESSED 